# DATA-2 — fragments B3 et S2 (DATA-1 amendé, pages et sha du manifeste de génération)

In [ ]:
import importlib, os, subprocess, sys, time, datetime, faulthandler
def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')
os.makedirs('/kaggle/working', exist_ok=True)
_TRACE = open('/kaggle/working/data2_cell1_trace.txt', 'a')
def _say(*parts):
    line = ' '.join(str(p) for p in parts)
    print(line, flush=True)
    _TRACE.write(line + '\n'); _TRACE.flush()
_say('data2 install start', _utc())
REQUIRED_OUTSIDE_IMAGE = []
for pkg in REQUIRED_OUTSIDE_IMAGE:
    _say('pip start', pkg, _utc())
    _t0 = time.monotonic()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--timeout', '30', '--retries', '1', pkg],
                   check=True, timeout=900)
    _say('pip end', pkg, _utc(), round(time.monotonic() - _t0, 1), 's')
faulthandler.dump_traceback_later(600, exit=True, file=_TRACE)
for mod in ["numpy", "torch", "PIL", "lxml"]:
    _say('import module', mod, _utc())
    importlib.import_module(mod)
_say('imports ok', _utc())
faulthandler.cancel_dump_traceback_later()


In [ ]:
import datetime, hashlib, importlib, importlib.util, json, os, platform, shutil, subprocess, sys, zipfile
from pathlib import Path

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

def _imp(name):
    print('import', name, _utc(), flush=True)
    return importlib.import_module(name)

print('data2 cell 2 start', _utc(), flush=True)
KIT_CANDIDATES = ["/kaggle/input/babaorum-data2-kit/scripts/data2_generate.py", "/kaggle/input/datasets/mabaku/babaorum-data2-kit/scripts/data2_generate.py"]
OUT = Path("/kaggle/working")
RAW = OUT / "gen_raw"
SRC = OUT / "heritage"
BUNDLE = OUT / "bundle"
GEN_OUT = OUT / "gen_out"
FRAG_OUT = OUT / "frag_out"
STATE = {}

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

E24_SHA_FULL = "714bc571db6a4e394027ec4e98cd4c997bde5ff54c735fbc8f32d9cdc4218654"
BUNDLE_V5_SHA = "ca41c09272248c5096f2547fe738eee622db5af50414182537ac8e94e43367f5"
OVERLAP_ACCEPT = {'accepted': True, 'by': 'pA', 'date': '2026-10-24', 'condition': 'fragments géométriques seulement, vérifié par test'}          # §14 (1)/(2): only OVERLAP_TEXT_FLAGGED, never OVERLAP_PAGE
OVERLAP_MENTION = "Les textes des lignes mfn sont presque tous identiques à des textes de lignes de mf_0000-0049 (même réservoir du générateur). Seules la géométrie et la mise en page sont neuves."
def check_reference(path):
    """The E2E-4 v3 report: whole sha256 and alphabet size 74 are checked before any use."""
    assert sha256(path) == E24_SHA_FULL, "E2E-4 report: sha256 differs from the frozen value"
    assert json.loads(Path(path).read_text())["alphabet_size"] == 74, "E2E-4 report: alphabet size is not 74"

_hits = [Path(c) for c in KIT_CANDIDATES if Path(c).is_file()]        # explicit candidates only, no listing
assert len(_hits) == 1, ("kit found %d times among the candidates" % len(_hits), [str(h) for h in _hits])
KIT_DIR = _hits[0].parent.parent
def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod
C = _load("data2_common", KIT_DIR / "scripts" / "data2_common.py")     # copied LD-0 functions (provenance in header)
print("kit", KIT_DIR, _utc(), flush=True)

GEN_MANIFEST = (lambda: STATE['found']['generation_manifest'])            # callable or path: the generation manifest (input or smoke output)

def d_controls():
    for m in ["numpy", "torch", "PIL", "lxml"]:
        _imp(m)

def d_discover():
    assert not (FRAG_OUT / "data1_manifest.json").exists(), "the output must start empty"
    found = C.discover_inputs(Path("/kaggle/input"), relpaths={"bundle": "stage_s_bundle_v5.zip.bin", "kit": "scripts/data1_fragments.py", "generation_manifest": "gen_out/data2_generation_manifest.json", "overlap_report": "gen_out/data2_overlap_report.json", "reference": "e2e4_extraction_report.json", "checkpoint_b3": "calibration/B3/checkpoint_last.pt", "checkpoint_s2": "calibration/S2/checkpoint_last.pt"})
    STATE["found"] = found
    print("discovery roots (names only):", found["roots_used"], flush=True)
    assert sha256(found["bundle"]) == BUNDLE_V5_SHA, "bundle v5 sha differs"
    check_reference(found["reference"])

def d_bundle_clean():
    shutil.rmtree(BUNDLE, ignore_errors=True)
    BUNDLE.mkdir(parents=True)

def d_kit_overlay():
    for rel in ["scripts/data1_fragments.py", "config/data1_fragments.json", "tests/test_data1_fragments.py", "scripts/data2_common.py", "scripts/data2_generate.py", "config/data2_pages.json", "tests/test_data2_generate.py", "scripts/data2_text_guard.py", "scripts/ld0_zone_coverage.py", "config/data2_smoke.json", "scripts/stage_s_notebooks/kaggle_data2_fragments.ipynb", "scripts/stage_s_notebooks/kaggle_data2_generation.ipynb", "scripts/stage_s_notebooks/kaggle_data2_smoke.ipynb", "tests/fixtures/data1_shard0/data1_manifest.json", "tests/fixtures/data1_shard0/LISEZMOI.txt", "tests/fixtures/data2_smoke_v3/data1_manifest.json", "scripts/stage_s_notebooks/build_kaggle_data1.py", "scripts/stage_s_notebooks/KAGGLE_INSTALL.txt", "scripts/stage_s_notebooks/build_kaggle_data2.py", "scripts/e2e3_fp_classes.py", "scripts/g_diag_mf.py", "scripts/e2e4_extraction.py", "config/e2e4_extraction.json"]:
        src = KIT_DIR / rel
        dst = BUNDLE / "babaorum" / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

def d_frag_check():
    gm = STATE["found"]["generation_manifest"] if "generation_manifest" in STATE["found"] else GEN_MANIFEST()
    manifest = json.loads(Path(gm).read_text())
    root = Path(gm).parent
    for key, want in manifest["files_sha256"].items():
        assert sha256(root / "mfn" / key) == want, ("generation file differs", key)
    STATE["gen_manifest"] = Path(gm)
    print("generation files checked:", len(manifest["files_sha256"]), flush=True)
    overlap_path = STATE['found']['overlap_report']
    overlap = json.loads(Path(overlap_path).read_text())
    st = overlap.get("status")
    if st != "OK" and not (st == "OVERLAP_TEXT_FLAGGED" and OVERLAP_ACCEPT.get("accepted") is True):
        raise RuntimeError("BLOCKED_OVERLAP: overlap report status is %s ; no fragment is produced" % st)
    FRAG_OUT.mkdir(parents=True, exist_ok=True)
    if st != "OK":
        (FRAG_OUT / "LISEZMOI_DATA2.txt").write_text(OVERLAP_MENTION + "\n", encoding="utf-8")
    STATE["overlap_status"] = st
    print("overlap report:", st, flush=True)

def d_sanitize():
    G2 = _load("data2_text_guard", KIT_DIR / "scripts" / "data2_text_guard.py")
    STATE["sanitize"] = G2.sanitize_file(FRAG_OUT / "data1_manifest.json")   # §15 (1)(2): original removed
    print("sanitize:", STATE["sanitize"]["removed_keys"], "original sha consigned", flush=True)

def d_text_guard():
    G2 = _load("data2_text_guard", KIT_DIR / "scripts" / "data2_text_guard.py")
    manifest = json.loads((FRAG_OUT / "data1_manifest.json").read_text())
    payload_files = [FRAG_OUT / "fragments" / arm / (pid + ".json")          # explicit paths, no listing
                     for arm in ("B3", "S2") for pid in manifest["pages_fragments"]]
    payload_files = [p for p in payload_files if p.is_file()]
    problems = G2.check_all([json.loads(p.read_text()) for p in payload_files], manifest)
    template = G2.template_from_source((KIT_DIR / "scripts" / "data1_fragments.py").read_text())
    cfg_e2e4 = json.loads((KIT_DIR / "config" / "e2e4_extraction.json").read_text())
    if not G2.readme_matches((FRAG_OUT / "LISEZMOI.txt").read_text(encoding="utf-8"), manifest, template, cfg_e2e4):
        problems.append("LISEZMOI.txt: texte différent du texte attendu")
    accepted = STATE.get("overlap_status") not in (None, "OK")
    if accepted and not G2.exact_text_matches((FRAG_OUT / "LISEZMOI_DATA2.txt").read_text(encoding="utf-8"),
                                              OVERLAP_MENTION + "\n"):
        problems.append("LISEZMOI_DATA2.txt: texte différent de la mention")
    extra = G2.unexpected_output(FRAG_OUT, G2.expected_output(manifest, accepted))
    if extra:
        problems.append("sortie hors liste: " + ", ".join(extra[:5]))
    if problems:
        for p in payload_files:
            p.unlink()
        raise RuntimeError("BLOCKED_TEXT_FIELD: " + "; ".join(problems[:5]))
    print("text guard: %d payload(s), manifest, LISEZMOI files and output list checked" % len(payload_files), flush=True)

def d_cleanup(rec):
    mpath = FRAG_OUT / "data2_notebook_manifest.json"
    FRAG_OUT.mkdir(parents=True, exist_ok=True)
    mani = {"exit_status": rec.get("status"), "durations_s": rec.get("durations_s"),
            "cleanup": {"bundle_removed": str(BUNDLE)},
            "overlap_status": STATE.get("overlap_status"),
            "original_manifest_sha256": (STATE.get("sanitize") or {}).get("original_sha256"),
            "clean_manifest_sha256": (STATE.get("sanitize") or {}).get("clean_sha256"),
            "overlap_text_mention": OVERLAP_MENTION if STATE.get("overlap_status") not in (None, "OK") else None}
    mpath.write_text(json.dumps(mani, indent=2))
    shutil.rmtree(BUNDLE, ignore_errors=True)

ENV = dict(os.environ, PYTHONPATH=str(BUNDLE / "babaorum" / "src") + os.pathsep + str(BUNDLE / "babaorum" / "scripts"),
           DATA2_WITH_TORCH="1")          # §11 (3): the DATA-2 test file has one DATA-1 test that imports torch
STEPS = [
    {"name": "controls", "fn": d_controls},
    {"name": "discover", "fn": d_discover},
    {"name": "bundle_clean", "fn": d_bundle_clean},
    {"name": "unzip", "argv": lambda: [sys.executable, "-c",
                                       "import sys, zipfile; zipfile.ZipFile(sys.argv[1]).extractall(sys.argv[2])",
                                       str(STATE["found"]["bundle"]), str(BUNDLE)], "timeout_s": 900},
    {"name": "kit_overlay", "fn": d_kit_overlay},
    {"name": "generation_check", "fn": d_frag_check},
    {"name": "pytest", "argv": lambda: [sys.executable, "-m", "pytest", "-q", "tests/test_data1_fragments.py",
                                        "tests/test_data2_generate.py"],
     "cwd": str(BUNDLE / "babaorum"), "env": lambda: ENV, "timeout_s": 900},
    {"name": "fragments", "argv": lambda: [sys.executable, "scripts/data1_fragments.py", "--confirmed-go",
                                           "--tmp", str(STATE["gen_manifest"].parent),
                                           "--calibration", str(STATE["found"]["checkpoint_b3"].parent.parent),
                                           "--generation-manifest", str(STATE["gen_manifest"]),
                                           "--out-dir", str(FRAG_OUT)] +
     (["--pages", ",".join([])] if [] else []),
     "cwd": str(BUNDLE / "babaorum"), "env": lambda: ENV, "timeout_s": 3600},
    {"name": "sanitize", "fn": d_sanitize},
    {"name": "text_guard", "fn": d_text_guard},
]

EXIT = C.execute_pipeline(OUT, STEPS, heartbeat_interval_s=60.0, cleanup=d_cleanup)
print("EXIT", json.dumps({k: EXIT[k] for k in ("status", "step", "exit_code")}), flush=True)
